In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
from rag_helper import RAGBase

instructions = """ 
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
"""

assistant = RAGBase(
    index=index, 
    llm_client=openai_client, 
    instructions=instructions
)

In [4]:
answer = assistant.rag('How do I run Ollama locally?')

print(answer)

To run Ollama locally:

1. Install Ollama from **https://ollama.com/download** for your operating system:
   - **macOS**: download and install the `.pkg`
   - **Windows**: download and install the `.msi`
   - **Linux**: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Open a terminal and run:
   ```bash
   ollama run llama3
   ```

This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

To check that the local server is running, you can also test:
```bash
curl http://localhost:11434
```

If needed in Python, install the client with:
```bash
pip install ollama
```


Working without tools and with tools

In [5]:
answer = assistant.rag('How do I run Olama locally?')

print(answer)

I don’t see any FAQ entry about **“Olama”** specifically.

If you mean **running the course locally**, the FAQ says you can do that if you’re comfortable setting up:

- Python
- `uv`
- Jupyter
- Docker
- any other tools needed for the module

Codespaces is just the easiest way to start with the same environment. If you run locally, make sure you **document your setup** and keep it **reproducible**.


In [6]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'Possibly — but it depends on whether the course is still open for enrollment.\n\nIf you want, I can help you figure it out quickly. Usually the key things are:\n\n- **Whether registration is still open**\n- **If the class has already started**\n- **Whether there’s a waitlist**\n- **Any prerequisites or approval needed**\n\nIf you’re contacting someone about it, you could say:\n\n> Hi, I just discovered the course and I’m very interested. Is it still possible to join?\n\nIf you want, I can also help you write a more polished message based on who you’re contacting.'

In [7]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query, 
        num_results=5, 
        boost_dict=boost_dict, 
        filter_dict=filter_dict
    )

In [8]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [9]:
response = openai_client.responses.create(
    model='gpt-5.4-mini', 
    input=messages, 
    tools=[search_tool]
)

In [10]:
len(response.output)

1

In [11]:
call = response.output[0]

In [12]:
call

ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I just discovered it? enrollment late join join course"}', call_id='call_QkR87DFUE2j8W3tnCNhNA9L3', name='search', type='function_call', id='fc_04f389acf7d885d1006a3a6a53ae588196b2f432af0b2a1932', namespace=None, status='completed')

In [13]:
import json

args = json.loads(call.arguments)
args

{'query': 'Can I join the course if I just discovered it? enrollment late join join course'}

In [14]:
call.name

'search'

In [15]:
results = search(**args)

In [16]:
result_json = json.dumps(results, indent=2)

In [17]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [18]:
messages.append(call)

In [19]:
messages.append(function_call_output)

In [20]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I just discovered it? enrollment late join join course"}', call_id='call_QkR87DFUE2j8W3tnCNhNA9L3', name='search', type='function_call', id='fc_04f389acf7d885d1006a3a6a53ae588196b2f432af0b2a1932', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_QkR87DFUE2j8W3tnCNhNA9L3',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "977bf7786c",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Course: I have registered for the LLM Zoomcamp. When can I e

In [21]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [22]:
print(response.output_text)

Yes — you can still join.

If you want a certificate, you’ll need to submit your project while submissions are still open.


In [23]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(777, 30)

In [24]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    # Prices per 1M tokens (example pricing)
    INPUT_PRICE_PER_MILLION = 0.15   # $0.15 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION = 0.60  # $0.60 / 1M output tokens

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }


# Your tokens
result = calculate_gpt54mini_price(652, 33)

print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 0.0001176


INSTRUCTIONS

In [25]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [64]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]


In [65]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [66]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment FAQ"}', call_id='call_tVwfJeplfIMfy7o4EHpDARTO', name='search', type='function_call', id='fc_0046e20b3363f2fa006a3a6efbf85c8195ad47d8956d703d2c', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course enrollment late join discovered course FAQ"}', call_id='call_XXY8OAye1Af9rQErJHCDq9fL', name='search', type='function_call', id='fc_0046e20b3363f2fa006a3a6efbf870819582038afbaf0cc069', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"can I join the course after start FAQ"}', call_id='call_dEJSjZlUiYFbihpjzgKoH6wm', name='search', type='function_call', id='fc_0046e20b3363f2fa006a3a6efbf87c8195b9155f49e5669355', namespace=None, status='completed')]

In [67]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        print(response.output[0].content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment FAQ"}
function_call: search {"query":"course enrollment late join discovered course FAQ"}
function_call: search {"query":"can I join the course after start FAQ"}


In [68]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [69]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        print(response.output[0].content[0].text)

ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still open. Also, for the certificate, the course needs to be completed with the live cohort rather than purely self-paced.

If you want, I can also help you with how to start the course or what to do first.


In [70]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment FAQ"}', call_id='call_tVwfJeplfIMfy7o4EHpDARTO', name='search', type='function_call', id='fc_0046e20b3363f2fa006a3a6efbf85c8195ad47d8956d703d2c', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course enrollment late join discovered course FAQ"}', call_id='call_XXY8OAye1Af9rQEr

Building the agent loop process

In [ ]:
def agent_loop(instructions, question, model='gpt-5.4-mini') -> str:
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1
    has_an_answer = False

    while has_an_answer is False:
        print(f'iteration #{it}...')

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)

            elif item.type == 'message':
                print('ASSISTANT found an answer')
                last_answer = item.content[0].text
                has_an_answer = True
        
        it += 1
    return last_answer


In [85]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches if required. First perform search, then analyze the results
and then perform more searchers if needed.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [86]:
result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment late registration FAQ"}
iteration #2...
ASSISTANT found an answer


In [88]:
print(result)

Yes — you can still join the course and start learning.

If you want a certificate, the important part is to submit your project while the course is still accepting submissions. Also, certificates are only available if you complete the course with a live cohort, not in self-paced mode.

If you'd like, I can also help you figure out how to start the course and what the weekly workflow looks like.


In [89]:
question = "what's queen gambit?"

result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"queen gambit chess opening queen's gambit what is it"}
iteration #2...
ASSISTANT found an answer


In [90]:
print(result)

The **Queen’s Gambit** is a **chess opening** that starts with:

1. **d4 d5**
2. **c4**

White offers the **c-pawn** as a “gambit” to try to gain control of the center and improve piece activity.  
It’s one of the most classical and popular openings in chess.

There are two main types:

- **Queen’s Gambit Accepted**: Black takes the c-pawn.
- **Queen’s Gambit Declined**: Black does not take it.

If you want, I can also explain **why it’s called a gambit** or show you the **basic ideas for both sides**.


In [91]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"

result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"Queen's Gambit opening chess course FAQ"}
iteration #3...
ASSISTANT found an answer


In [92]:
print(result)

I couldn’t find anything in the course FAQ about “queen gambit,” so it looks like this isn’t a course/logistics question.

If you meant something course-related, feel free to rephrase, and I can check the FAQ again. Are there other areas you want to explore?


FRAMEWORK

In [93]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [103]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [100]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={'question': 3.0, 'section': 0.5},
        filter_dict={'course': 'llm-zoomcamp'}
    )

In [105]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [106]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [110]:
chat_interface = IPythonChatInterface()
callback=DisplayingRunnerCallback(chat_interface)

In [111]:
runner = OpenAIResponsesRunner(
    tools=agent_tools, 
    developer_prompt=instructions, 
    chat_interface=chat_interface, 
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [ ]:
result = runner.loop(
    prompt='How do I run Olama?', 
    callback=callback
)

-> Response received


-> Response received


-> Response received


In [117]:
result.cost

CostInfo(input_cost=Decimal('0.00272325'), output_cost=Decimal('0.000846'), total_cost=Decimal('0.00356925'))

In [118]:
result.all_messages

[EasyInputMessage(content="\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searchers. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.\n", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama run Ollama how do I run Ollama course FAQ"}', call_id='call

In [120]:
result = runner.loop(
    prompt='How do I run Olama a different model?', 
    previous_messages=result.all_messages, 
    callback=callback
)

-> Response received


-> Response received


In [121]:
result.cost

CostInfo(input_cost=Decimal('0.004362'), output_cost=Decimal('0.0006795'), total_cost=Decimal('0.0050415'))

In [122]:
runner.run()

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searchers. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.\n", role='developer', phase=None, type=None), EasyInputMessage(content='How can I approve the course?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"approve the course course approva